In [2]:
import qiskit
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
import qiskit_aer.noise as noise
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.visualization import plot_histogram
from qiskit_aer.noise import (NoiseModel, pauli_error, depolarizing_error, thermal_relaxation_error, ReadoutError)


In [ ]:
x_error_prob = 2.668e-4
sx_error_prob = 2.668E-4
cz_error_prob = 1.961e-3
id_error_prob = 2.668e-4
init_error_prob = 2.000e-4
T1 = 232.01 #us
T2 = 136.04 #us
single_qubit_gate_length = 32 #ns
#two_qubit_gate_length = 
prob_meas0_prep1 = 0.01196
prob_meas1_prep0 = 0.00696
#z_error is implemented as microwave pulses(it saves the phase information of the qubit cos(wt + phi))

noise_model = noise.NoiseModel()

x_error  = noise.depolarizing_error(2*x_error_prob, 1)
sx_error = noise.depolarizing_error(2*sx_error_prob, 1)
id_error = noise.depolarizing_error(2*id_error_prob, 1)
cz_error = noise.depolarizing_error(4/3*cz_error_prob, 2)
init_error = noise.depolarizing_error(init_error_prob, 1)
noise_model.add_all_qubit_quantum_error(x_error, ['x'])
noise_model.add_all_qubit_quantum_error(sx_error, ['sx'])
noise_model.add_all_qubit_quantum_error(cz_error, ['cz'])
noise_model.add_all_qubit_quantum_error(id_error, ['id'])

readout_error = ReadoutError([
    [1 - prob_meas1_prep0, prob_meas1_prep0],   # 실제 |0⟩
    [prob_meas0_prep1, 1 - prob_meas0_prep1],   # 실제 |1⟩
])
noise_model.add_all_qubit_readout_error(readout_error)


In [12]:
sim_noise = AerSimulator(noise_model=noise_model)

def bell_chain(n):
    qc = QuantumCircuit(2)
    qc.h(0); qc.cx(0, 1); qc.barrier()
    for _ in range(n):
        qc.cx(0, 1); qc.h(0); qc.barrier()   
        qc.h(0); qc.cx(0, 1); qc.barrier()   
    qc.measure_all()
    return qc

for n in range(1000):                     
    tqc = transpile(bell_chain(n), basis_gates=["cz", "sx", "rz", "x", "id"])
    print(sim_noise.run(tqc, shots=10000).result().get_counts())
    

{'01': 98, '10': 108, '00': 4977, '11': 4817}
{'01': 119, '10': 117, '11': 4849, '00': 4915}
{'01': 152, '10': 138, '00': 4816, '11': 4894}
{'01': 156, '10': 191, '00': 4806, '11': 4847}
{'01': 191, '10': 164, '11': 4835, '00': 4810}
{'01': 210, '10': 191, '00': 4803, '11': 4796}
{'01': 212, '10': 210, '11': 4883, '00': 4695}
{'01': 240, '10': 204, '00': 4693, '11': 4863}
{'01': 239, '10': 253, '11': 4789, '00': 4719}
{'01': 281, '10': 282, '00': 4723, '11': 4714}
{'01': 277, '10': 282, '11': 4734, '00': 4707}
{'01': 291, '10': 274, '11': 4800, '00': 4635}
{'01': 261, '10': 305, '00': 4710, '11': 4724}
{'01': 321, '10': 363, '11': 4685, '00': 4631}
{'01': 323, '10': 336, '11': 4674, '00': 4667}
{'01': 383, '10': 352, '00': 4677, '11': 4588}
{'01': 364, '11': 4522, '00': 4740, '10': 374}
{'01': 401, '10': 410, '00': 4670, '11': 4519}
{'01': 408, '10': 430, '11': 4491, '00': 4671}
{'01': 406, '00': 4592, '10': 419, '11': 4583}
{'01': 483, '10': 434, '00': 4542, '11': 4541}
{'01': 435, '1

In [11]:
print(sim_noise.options.noise_model)

NoiseModel:
  Basis gates: ['cx', 'cz', 'id', 'rz', 'sx', 'x']
  Instructions with noise: ['id', 'x', 'cz', 'sx', 'measure']
  All-qubits errors: ['x', 'sx', 'cz', 'id', 'measure']


In [7]:
sim = AerSimulator(method="density_matrix", noise_model=noise_model)
def bell_chain(n):
    qc = QuantumCircuit(2)
    qc.h(0); qc.cx(0, 1); qc.barrier()
    for _ in range(n):
        qc.cx(0, 1); qc.h(0); qc.barrier()   
        qc.h(0); qc.cx(0, 1); qc.barrier()   
    qc.measure_all()
    return qc
tqc = transpile(bell_chain(30), basis_gates=["cz","sx","rz","x","id"])
tqc.remove_final_measurements(); tqc.save_probabilities()
print(sim.run(tqc).result().data()["probabilities"])

[0.44966497 0.05033503 0.05033503 0.44966497]
